# Módulo 01: Introducción a Modelos Definidos por el Usuario (`gsua_csb`)
**Curso guiado de GSUA-CSB**

En esta guía de laboratorio interactiva se estudian dos paradigmas de modelación dinámica:
1. **Tiempo Discreto**: Modelo de Crecimiento Logístico ($N_{t+1} = N_t + r N_t (1 - N_t / K)$).
2. **Tiempo Continuo**: Modelo Epidémico SIR ($dS/dt, dI/dt, dR/dt$) resuelto con `scipy.integrate.solve_ivp`.

Se especifican explícitamente tanto los **factores** (parámetros del sistema y condiciones iniciales) como los **nombres de las salidas** del modelo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gsua_csb import design_matrix
from discrete_logistic_userdefined import create_discrete_logistic_model
from sir_continuous_userdefined import create_sir_continuous_model

## 1. Modelo en Tiempo Discreto: Crecimiento Logístico

Definimos el modelo especificando los factores (parámetros $r, K$ y la condición inicial $N_0$).

In [ ]:
model_discrete = create_discrete_logistic_model()

print(f"Factores (Parámetros + C.I.): {model_discrete.names}")
print(f"Límites de los factores:\n{model_discrete.range}")
print(f"Punto nominal: {model_discrete.nominal}")

# Simulación nominal
y_discrete_nom = np.squeeze(model_discrete.evaluate(model_discrete.nominal, model_discrete.domain))

# Muestreo de ensamble (25 trayectorias) usando design_matrix
M_discrete = design_matrix(model_discrete, n=25, method="latin_hypercube", seed=42)
Y_discrete_ens = model_discrete.evaluate_batch(M_discrete, model_discrete.domain)

# Visualización del ensamble
plt.figure(figsize=(9, 4.5))
for i in range(25):
    plt.plot(model_discrete.domain, Y_discrete_ens[i], color='lightblue', alpha=0.6, linewidth=0.8)
plt.plot(model_discrete.domain, y_discrete_nom, color='tab:blue', linewidth=2.5, label='Nominal')
plt.xlabel('Paso de Tiempo (t)')
plt.ylabel('Población N(t)')
plt.title('Crecimiento Logístico Discreto: Ensamble de 25 Trayectorias')
plt.grid(True)
plt.legend()
plt.show()

## 2. Modelo en Tiempo Continuo: SIR (Susceptible - Infectado - Recuperado)

Definimos el modelo continuo declarando los factores `['beta', 'gamma', 'S0', 'I0']` y las 3 salidas `['Susceptibles', 'Infectados', 'Recuperados']`.

In [ ]:
model_sir = create_sir_continuous_model(output_selection=None) # Exponer las 3 salidas

print(f"Factores (Parámetros + C.I.): {model_sir.names}")
print(f"Nombres de salidas: {model_sir.output_names}")

# Simulación nominal
y_sir_all = model_sir.evaluate(model_sir.nominal, model_sir.domain)
S_nom, I_nom, R_nom = y_sir_all[0], y_sir_all[1], y_sir_all[2]

beta_nom, gamma_nom, S0_nom, I0_nom = model_sir.nominal
R0_basic = (beta_nom * S0_nom) / gamma_nom
print(f"Número Reproductivo Básico R0 = {R0_basic:.2f}")

# Gráfica de evolución temporal y plano de fase (S vs I)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(model_sir.domain, S_nom, color='tab:blue', linewidth=2, label='S(t) Susceptibles')
axes[0].plot(model_sir.domain, I_nom, color='tab:red', linewidth=2, label='I(t) Infectados')
axes[0].plot(model_sir.domain, R_nom, color='tab:green', linewidth=2, label='R(t) Recuperados')
axes[0].set_xlabel('Tiempo (Días)')
axes[0].set_ylabel('Población (Personas)')
axes[0].set_title(f'Evolución Temporal SIR (R0 = {R0_basic:.2f})')
axes[0].grid(True)
axes[0].legend()

axes[1].plot(S_nom, I_nom, color='purple', linewidth=2.5)
axes[1].plot(S_nom[0], I_nom[0], 'ko', markersize=8, label=f'Inicio ({S0_nom:.0f}, {I0_nom:.0f})')
axes[1].set_xlabel('Susceptibles S(t)')
axes[1].set_ylabel('Infectados I(t)')
axes[1].set_title('Plano de Fase Epidemiológico (S vs I)')
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

### 2.1 Ensamble de Trayectorias de Infectados frente a Incertidumbre
Seleccionamos únicamente la salida de Infectados (`set_output("Infectados")`) y evaluamos un muestreo de Hipercubo Latino de 30 combinaciones.

In [ ]:
model_sir.set_output("Infectados")
M_sir = design_matrix(model_sir, n=30, method="latin_hypercube", seed=42)
Y_sir_ens = model_sir.evaluate_batch(M_sir, model_sir.domain)
if Y_sir_ens.ndim == 3:
    Y_sir_ens = Y_sir_ens.squeeze(axis=1)

plt.figure(figsize=(9, 4.5))
for i in range(30):
    plt.plot(model_sir.domain, Y_sir_ens[i], color='pink', alpha=0.6, linewidth=0.8)
plt.plot(model_sir.domain, I_nom, color='tab:red', linewidth=2.5, label='Nominal')
plt.xlabel('Tiempo (Días)')
plt.ylabel('Número de Infectados I(t)')
plt.title('Variabilidad de la Curva de Infectados frente a Incertidumbre en Factores')
plt.grid(True)
plt.legend()
plt.show()

## 3. Preguntas de Análisis para el Estudiante

1. **Tiempo de Pico Epidémico**: Modifique el parámetro $\beta$ aumentando su valor en un $20\%$. ¿Cómo se desplaza el tiempo del pico $t_{max}$ y la magnitud máxima $I_{max}$?
2. **Efecto de la Condición Inicial**: Manteniendo $\beta$ y $\gamma$ fijos, altere $I_0$ de 1 a 50. ¿Varía el número total final de susceptibles que escapan a la infección?
3. **Caos en Tiempo Discreto**: En el modelo logístico discreto, cambie el parámetro $r$ a $r = 2.2$ y $r = 2.7$. Observe y describa el comportamiento cualitativo de la población.